## Connect to Wharton Research Data Services (WRDS)

In [14]:
import wrds
import os
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
db = wrds.Connection(wrds_username=os.getenv("WRDS_USERNAME"))

In [ ]:
libraries = db.list_libraries()

In [ ]:
crsp = db.list_tables(library='crsp')

In [ ]:
dsf_info = db.describe_table(library="crsp", table="dsf")
dsf_info

Approximately 107682104 rows in crsp.dsf.


,name,nullable,type,comment
0,cusip,True,VARCHAR(8),CUSIP Header
1,permno,True,INTEGER,PERMNO
2,permco,True,INTEGER,PERMCO
3,issuno,True,INTEGER,Nasdaq Issue Number
4,hexcd,True,SMALLINT,Exchange Code Header
5,hsiccd,True,INTEGER,Standard Industrial Classification Code Header
6,date,True,DATE,Date of Observation
7,bidlo,True,"NUMERIC(11, 5)",Bid or Low Price
8,askhi,True,"NUMERIC(11, 5)",Ask or High Price
9,prc,True,"NUMERIC(11, 5)",Price or Bid/Ask Average


In [ ]:
dsfv2_info = db.describe_table(library="crsp", table="dsf_v2")
dsfv2_info

Approximately 110257376 rows in crsp.dsf_v2.


,name,nullable,type,comment
0,permno,True,INTEGER,PERMNO
1,hdrcusip,True,VARCHAR(8),Header CUSIP -8 Characters
2,permco,True,INTEGER,PERMCO
3,siccd,True,INTEGER,Sic Code
4,nasdissuno,True,INTEGER,Nasdaq Issue Number
5,yyyymmdd,True,INTEGER,YYYYMMDD - Daily Calendar Period Key
6,sharetype,True,VARCHAR(3),Share Type
7,securitytype,True,VARCHAR(4),Security Type
8,securitysubtype,True,VARCHAR(3),Security Sub-Type
9,usincflg,True,VARCHAR(1),US Incorporation Flag


### Get S&P500 Constituents

* Get all S&P constituents since 1957-03-04, when S&P 500 is officially created.
* Get their daily price in the time being.

(Note on SQL: inner join daily price of each stock in the given S&P500 perio, left join their common name / ticker)

In [ ]:
sp500 = db.raw_sql("""
    SELECT
        a.*,
        b.comnam,
        b.namedt,
        b.nameendt,
        b.ticker,
        c.dlycaldt AS date,
        c.dlyprc

    FROM crsp.msp500list AS a

    INNER JOIN crsp.dsf_v2 AS c
        ON a.permno = c.permno
        AND c.dlycaldt BETWEEN a.start AND a.ending
       

    LEFT JOIN crsp.msenames AS b
        ON a.permno = b.permno
        AND c.dlycaldt BETWEEN b.namedt AND b.nameendt

    WHERE c.dlycaldt >= DATE '1957-03-04'
    
    ORDER BY c.dlycaldt, a.permno
""",
date_cols=[
    "start",
    "ending",
    "namedt",
    "nameendt",
    "date"
])

sp500

,permno,start,ending,comnam,namedt,nameendt,ticker,date,dlyprc
0,10006,1957-03-01,1984-07-18,A C F INDUSTRIES INC,1954-06-01,1962-07-01,<NA>,1957-03-04,60.75
1,10030,1957-03-01,1969-01-08,AMERICAN BRAKE SHOE CO,1943-04-27,1962-07-01,<NA>,1957-03-04,43.125
2,10057,1957-03-01,1992-07-02,NATIONAL ACME CO,1925-12-31,1962-07-01,<NA>,1957-03-04,73.0
3,10102,1957-03-01,1975-02-05,AIR REDUCTION INC,1925-12-31,1962-07-01,<NA>,1957-03-04,52.5
4,10137,1925-12-31,1976-06-30,WEST PENN ELECTRIC CO,1948-01-16,1960-11-09,<NA>,1957-03-04,26.25
...,...,...,...,...,...,...,...,...,...
44786,93096,2012-12-03,2024-12-31,DOLLAR GENERAL CORP NEW,2024-03-12,2024-12-31,DG,2024-12-31,75.82
44787,93132,2018-10-11,2024-12-31,FORTINET INC,2019-08-02,2024-12-31,FTNT,2024-12-31,94.48
44788,93246,2021-03-22,2024-12-31,GENERAC HOLDINGS INC,2010-02-11,2024-12-31,GNRC,2024-12-31,155.05
44789,93429,2017-03-01,2024-12-31,C B O E GLOBAL MARKETS INC,2024-06-21,2024-12-31,CBOE,2024-12-31,195.4
